![DB Academy](./Includes/images/common/db-academy.png)

# Demo - Data Setup and Exploration

## Overview

In this demo, you will explore the source data for the Lakeflow Designer workflow you will build in the next notebook.

You will use Catalog Explorer to inspect three bronze tables (sessions, participants, feedback) and preview a CSV file that you will later upload through Lakeflow Designer.

By the end, you will know the shape, details, and join keys of every dataset, and you will understand the medallion architecture, **bronze -> silver -> gold** workflow you build next.

## Learning Objectives

By the end of this demo, you will be able to:

- Navigate Catalog Explorer to inspect schemas, sample data, and table metadata.
- Identify the details and join keys for each bronze table.
- Recognize nested data (an `array<struct>`) and explain why it must be flattened in silver.
- Describe the medallion architecture you will build in Lakeflow Designer.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**

![Serverless Select](./Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:** This notebook requires **SQL and Python, and was tested using Serverless V5**. All purpose compute may work but is not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


## A. Classroom Setup

1. Run the cell below. It will:

    - Create (or use) your personal catalog named **learn_no_code_etl_designer_USERNAME**.
    - Create the **designer_meeting** schema.
    - Create the necessary bronze tables.

In [0]:
%run ./Includes/Classroom-Setup-1

ℹ️ Non-Vocareum workspace detected. Setting up catalog.
ℹ️ Using default catalog name: 'learn_no_code_etl_designer_puli_reddy'.
ℹ️ Catalog 'learn_no_code_etl_designer_puli_reddy' not found. Creating it now...
✅ Catalog 'learn_no_code_etl_designer_puli_reddy' created successfully.


DataFrame[]


  STEP 1: Verifying catalog exists: learn_no_code_etl_designer_puli_reddy
  Catalog 'learn_no_code_etl_designer_puli_reddy' exists.

  STEP 2: Setting up 1 schema(s) in catalog: learn_no_code_etl_designer_puli_reddy
  [1/1] Checking: `learn_no_code_etl_designer_puli_reddy`.`designer_meeting`... CREATED

  COMPLETE: 1 schema(s) created, 0 already existed.




  Searching for 'Includes/data' folder...
  Current directory: /Workspace/Users/puli.reddy@gmail.com/No Code ETL with Lakeflow Designer
  Checking: /Workspace/Users/puli.reddy@gmail.com/No Code ETL with Lakeflow Designer/Includes/data... FOUND




  STEP 1: Validating source workspace folder...
  Source folder found: /Workspace/Users/puli.reddy@gmail.com/No Code ETL with Lakeflow Designer/Includes/data/feedback

  STEP 2: Checking target volume path...
  Creating subdirectory within volume...
  Created: /Volumes/learn_no_code_etl_designer_puli_reddy/designer_meeting/source_files/feedback

  STEP 3: Reading source files...
  Found 1 file(s) in source folder.

  STEP 4: Copying 1 file(s) to target volume...
  Source:      /Workspace/Users/puli.reddy@gmail.com/No Code ETL with Lakeflow Designer/Includes/data/feedback
  Destination: /Volumes/learn_no_code_etl_designer_puli_reddy/designer_meeting/source_files/feedback
  [1/1] Checking: how_feedback_week_1.json... NOT found at destination. Copied successfully.

  COMPLETE: Copied 1 new file(s), skipped 0.


  STEP 1: Validating source workspace folder...
  Source folder found: /Workspace/Users/puli.reddy@gmail.com/No Code ETL with Lakeflow Designer/Includes/data/participants

  STEP 

Creating learn_no_code_etl_designer_puli_reddy.designer_meeting.participants_bronze from /Volumes/learn_no_code_etl_designer_puli_reddy/designer_meeting/source_files/participants

CREATE TABLE learn_no_code_etl_designer_puli_reddy.designer_meeting.participants_bronze AS
SELECT * 
FROM read_files('/Volumes/learn_no_code_etl_designer_puli_reddy/designer_meeting/source_files/participants', format => 'json', multiLine => 'true'
)
Done. Rows: 3,139
Creating learn_no_code_etl_designer_puli_reddy.designer_meeting.feedback_bronze from /Volumes/learn_no_code_etl_designer_puli_reddy/designer_meeting/source_files/feedback

CREATE TABLE learn_no_code_etl_designer_puli_reddy.designer_meeting.feedback_bronze AS
SELECT * 
FROM read_files('/Volumes/learn_no_code_etl_designer_puli_reddy/designer_meeting/source_files/feedback', format => 'json', multiLine => 'true', schema => 'meeting_uuid STRING, responses STRING'
)
Done. Rows: 40
Creating learn_no_code_etl_designer_puli_reddy.designer_meeting.sessions

Information,Value
Your Catalog:,learn_no_code_etl_designer_puli_reddy
Your Schema:,designer_meeting


Compute,Status,Details
Serverless,✓ Match,Version 5


2. Run the cell below and confirm the following tables were created in your **learn_no_code_etl_designer_USERNAME.designer_meeting** schema:
    - **feedback_bronze**
    - **participants_bronze**
    - **sessions_bronze**

In [0]:
%sql
SHOW TABLES IN designer_meeting;

database,tableName,isTemporary
designer_meeting,feedback_bronze,false
designer_meeting,participants_bronze,false
designer_meeting,sessions_bronze,false


## B. Scenario Overview

### B1. Medallion Architecture Overview
<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026; text-align: center;">

<div style="font-size: 22pt; font-weight: 700; margin-bottom: 6px;">Medallion Architecture</div>
<div style="font-size: 14pt; color: #618794; margin-bottom: 22px;">Progressively refining data through Bronze, Silver, and Gold layers</div>

<!-- MAIN FLOW (CENTERED) -->
<div style="display: flex; justify-content: center; align-items: stretch; gap: 16px;">

  <!-- BRONZE BOX -->
  <div style="flex: 0 0 220px; background: #F9F7F4; border-radius: 10px; padding: 18px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; border-top: 6px solid #CD7F32; display: flex; flex-direction: column;">
    <svg width="56" height="46" viewBox="0 0 56 46" xmlns="http://www.w3.org/2000/svg" style="margin: 0 auto;">
      <rect x="2" y="2" width="52" height="42" rx="4" fill="none" stroke="#CD7F32" stroke-width="2"/>
      <rect x="2" y="2" width="52" height="12" fill="#CD7F32"/>
      <line x1="2" y1="24" x2="54" y2="24" stroke="#CD7F32" stroke-width="1.5"/>
      <line x1="2" y1="34" x2="54" y2="34" stroke="#CD7F32" stroke-width="1.5"/>
      <line x1="20" y1="14" x2="20" y2="44" stroke="#CD7F32" stroke-width="1.5"/>
      <line x1="38" y1="14" x2="38" y2="44" stroke="#CD7F32" stroke-width="1.5"/>
    </svg>
    <div style="font-size: 16pt; font-weight: 700; margin-top: 10px; color: #CD7F32;">Bronze</div>
    <div style="font-size: 14pt; color: #618794; margin-bottom: 8px;">Raw, as ingested</div>
    <ul style="font-size: 14pt; padding-left: 18px; margin: 10px 0 0 0; text-align: left; flex-grow: 1;">
      <li style="margin-bottom: 6px;">Source files and streams</li>
      <li style="margin-bottom: 6px;">Schema preserved</li>
      <li>Full history kept</li>
    </ul>
  </div>

  <!-- ARROW -->
  <div style="font-size: 28pt; color: #618794; display: flex; align-items: center;">&rarr;</div>

  <!-- SILVER BOX -->
  <div style="flex: 0 0 220px; background: #F9F7F4; border-radius: 10px; padding: 18px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; border-top: 6px solid #A8B5BC; display: flex; flex-direction: column;">
    <svg width="56" height="46" viewBox="0 0 56 46" xmlns="http://www.w3.org/2000/svg" style="margin: 0 auto;">
      <rect x="2" y="2" width="52" height="42" rx="4" fill="none" stroke="#A8B5BC" stroke-width="2"/>
      <rect x="2" y="2" width="52" height="12" fill="#A8B5BC"/>
      <line x1="2" y1="24" x2="54" y2="24" stroke="#A8B5BC" stroke-width="1.5"/>
      <line x1="2" y1="34" x2="54" y2="34" stroke="#A8B5BC" stroke-width="1.5"/>
      <line x1="20" y1="14" x2="20" y2="44" stroke="#A8B5BC" stroke-width="1.5"/>
      <line x1="38" y1="14" x2="38" y2="44" stroke="#A8B5BC" stroke-width="1.5"/>
    </svg>
    <div style="font-size: 16pt; font-weight: 700; margin-top: 10px; color: #5A6F77;">Silver</div>
    <div style="font-size: 14pt; color: #618794; margin-bottom: 8px;">Cleaned, conformed</div>
    <ul style="font-size: 14pt; padding-left: 18px; margin: 10px 0 0 0; text-align: left; flex-grow: 1;">
      <li style="margin-bottom: 6px;">Validated and typed</li>
      <li style="margin-bottom: 6px;">Joined and deduplicated</li>
      <li>Ready for modeling</li>
    </ul>
  </div>

  <!-- ARROW -->
  <div style="font-size: 28pt; color: #618794; display: flex; align-items: center;">&rarr;</div>

  <!-- GOLD BOX -->
  <div style="flex: 0 0 220px; background: #F9F7F4; border-radius: 10px; padding: 18px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; border-top: 6px solid #FFAB00; display: flex; flex-direction: column;">
    <svg width="56" height="46" viewBox="0 0 56 46" xmlns="http://www.w3.org/2000/svg" style="margin: 0 auto;">
      <rect x="2" y="2" width="52" height="42" rx="4" fill="none" stroke="#FFAB00" stroke-width="2"/>
      <rect x="2" y="2" width="52" height="12" fill="#FFAB00"/>
      <line x1="2" y1="24" x2="54" y2="24" stroke="#FFAB00" stroke-width="1.5"/>
      <line x1="2" y1="34" x2="54" y2="34" stroke="#FFAB00" stroke-width="1.5"/>
      <line x1="20" y1="14" x2="20" y2="44" stroke="#FFAB00" stroke-width="1.5"/>
      <line x1="38" y1="14" x2="38" y2="44" stroke="#FFAB00" stroke-width="1.5"/>
    </svg>
    <div style="font-size: 16pt; font-weight: 700; margin-top: 10px; color: #B07700;">Gold</div>
    <div style="font-size: 14pt; color: #618794; margin-bottom: 8px;">Business ready</div>
    <ul style="font-size: 14pt; padding-left: 18px; margin: 10px 0 0 0; text-align: left; flex-grow: 1;">
      <li style="margin-bottom: 6px;">Aggregated metrics</li>
      <li style="margin-bottom: 6px;">Curated for analytics</li>
      <li>Powers BI and AI</li>
    </ul>
  </div>

</div>

<!-- BOTTOM CALLOUT -->
<div style="margin-top: 22px; padding: 16px 20px; background: #F9F7F4; border: 2px solid #1B5162; border-radius: 8px; font-size: 14pt; text-align: left;">
  <b>Each layer adds quality and value.</b> Bronze captures the source of truth, Silver cleans and conforms it for reuse, and Gold serves the trusted, analytics ready tables that power dashboards, reports, and AI.
</div>

</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>

### Why a layered approach?

- Data quality improves at each step, so downstream users always work with the version that matches their needs.
- Each layer has a clear job, which keeps data flows easier to debug, govern, and evolve over time.
- Reprocessing is safe. If a bug is found in Silver or Gold logic, you can rebuild from Bronze without re-pulling from the source.

### Bronze: raw, as ingested

- The landing zone for data coming from files, databases, APIs, or streams.
- Stored close to the original format. Schema is preserved and bad records are kept rather than dropped.
- Use Bronze when you need a faithful history of what arrived and when.
- Typical operations: ingest with Auto Loader, append new files, add ingestion timestamps.

### Silver: cleaned and conformed

- The working layer where data becomes trustworthy.
- Types are enforced, duplicates are removed, keys are standardized, and tables are joined into a coherent model.
- Use Silver when you want a reliable foundation for analysts, data scientists, and downstream workflows.
- Typical operations: cast types, drop or quarantine bad rows, deduplicate on keys, join reference data.

### Gold: business ready

- The serving layer purpose built for consumption.
- Tables are aggregated, modeled, and named for the business question they answer.
- Use Gold when you want fast, governed, easy to understand tables for dashboards, reports, and AI features.
- Typical operations: group and aggregate, compute KPIs, build star schemas or feature tables.

### How it fits Lakeflow Designer

- In this workshop the Bronze tables are already created for you.
- You will build Silver transformations in Lakeflow Designer to clean and join the data.
- Then you will build Gold tables that answer real questions about sessions, participants, and feedback.

</details>


### B2. Scenario
<br></br>
<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<style>
.sc-header {
    background: #1B5162;
    color: white;
    border-radius: 10px;
    padding: 22px 26px;
    text-align: center;
    margin-bottom: 18px;
}
.sc-header-title {
    font-size: 20pt;
    font-weight: 700;
    margin: 0 0 8px 0;
}
.sc-header-sub {
    font-size: 14pt;
    line-height: 1.5;
    margin: 0;
    opacity: 0.92;
}

.sc-cards {
    display: flex;
    flex-direction: column;
    gap: 12px;
    margin-bottom: 18px;
}

.sc-card {
    background: #F9F7F4;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    padding: 16px 18px 16px 26px;
    position: relative;
    box-sizing: border-box;
    display: grid;
    grid-template-columns: 48px 1fr 1fr;
    gap: 16px;
    align-items: center;
}

.sc-card::before {
    content: "";
    position: absolute;
    top: 0;
    left: 0;
    width: 8px;
    height: 100%;
    border-top-left-radius: 8px;
    border-bottom-left-radius: 8px;
}

.sc-card.bronze::before { background: #CD7F32; }
.sc-card.csv::before    { background: #00A972; }

.sc-icon-box {
    width: 42px;
    height: 42px;
    border-radius: 8px;
    display: flex;
    align-items: center;
    justify-content: center;
    color: white;
    position: relative;
}

.sc-card.bronze .sc-icon-box { background: #CD7F32; }
.sc-card.csv .sc-icon-box    { background: #00A972; }

.sc-icon-box svg {
    width: 24px;
    height: 24px;
}

.sc-csv-badge {
    position: absolute;
    bottom: 2px;
    right: 2px;
    background: white;
    color: #00A972;
    font-size: 7pt;
    font-weight: 700;
    padding: 1px 3px;
    border-radius: 2px;
    line-height: 1;
}

.sc-card-name {
    font-size: 14pt;
    font-weight: 700;
    color: #0b2026;
    margin: 0 0 4px 0;
    word-break: break-word;
}

.sc-card-source {
    font-size: 11pt;
    color: #5E7077;
    font-style: italic;
    margin: 0 0 6px 0;
}

.sc-card-cols {
    font-size: 10.5pt;
    color: #5E7077;
    line-height: 1.4;
    margin: 0;
}

.sc-card-meta {
    font-size: 11pt;
    color: #0b2026;
    line-height: 1.55;
}

.sc-card-meta strong {
    color: #0b2026;
}

.sc-job {
    padding: 18px 24px;
    background: #FFF6F4;
    border: 2px solid #FF5F46;
    border-radius: 10px;
}
.sc-job-title {
    font-size: 14pt;
    font-weight: 700;
    color: #0b2026;
    margin: 0 0 10px 0;
}
.sc-job-list {
    font-size: 13pt;
    color: #0b2026;
    line-height: 1.55;
    padding-left: 20px;
    margin: 0;
}
.sc-job-list li {
    margin-bottom: 8px;
}
.sc-job-list li:last-child {
    margin-bottom: 0;
}
</style>

<div class="sc-header">
  <div class="sc-header-title">The Scenario: Weekly Event Operations Data</div>
  <p class="sc-header-sub">A virtual events team runs weekly "Hands on Workshop (HOW)" deep-dive sessions and collects four raw datasets every week. Your job is to turn that raw data into something the business can actually use.</p>
</div>

<div class="sc-cards">

  <div class="sc-card bronze">
    <div class="sc-icon-box">
      <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><rect x="3" y="3" width="18" height="18" rx="2"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="15" x2="21" y2="15"/><line x1="9" y1="3" x2="9" y2="21"/><line x1="15" y1="3" x2="15" y2="21"/></svg>
    </div>
    <div>
      <div class="sc-card-name">sessions_bronze</div>
      <div class="sc-card-source">Delta table from events platform</div>
      <p class="sc-card-cols"><strong>Columns:</strong> meeting_uuid, how_name, event_date, presenter, topic_category, difficulty_level, planned_duration_minutes</p>
    </div>
    <div class="sc-card-meta">
      <strong>Grain:</strong> one row per session<br>
      <strong>Rows:</strong> 41
    </div>
  </div>

  <div class="sc-card bronze">
    <div class="sc-icon-box">
      <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><rect x="3" y="3" width="18" height="18" rx="2"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="15" x2="21" y2="15"/><line x1="9" y1="3" x2="9" y2="21"/><line x1="15" y1="3" x2="15" y2="21"/></svg>
    </div>
    <div>
      <div class="sc-card-name">participants_bronze</div>
      <div class="sc-card-source">Delta table from events platform</div>
      <p class="sc-card-cols"><strong>Columns:</strong> meeting_uuid, id, join_time, leave_time</p>
    </div>
    <div class="sc-card-meta">
      <strong>Grain:</strong> one row per attendee, per session<br>
      <strong>Rows:</strong> 3,139
    </div>
  </div>

  <div class="sc-card bronze">
    <div class="sc-icon-box">
      <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><rect x="3" y="3" width="18" height="18" rx="2"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="15" x2="21" y2="15"/><line x1="9" y1="3" x2="9" y2="21"/><line x1="15" y1="3" x2="15" y2="21"/></svg>
    </div>
    <div>
      <div class="sc-card-name">feedback_bronze</div>
      <div class="sc-card-source">Delta table, contains nested JSON</div>
      <p class="sc-card-cols"><strong>Columns:</strong> meeting_uuid, responses <span style="font-family: monospace; color: #98182A; background: #FFF6F4; padding: 1px 6px; border-radius: 4px;">JSON array</span><br><strong>Nested fields:</strong> answer, question, question_id, respondent_id</p>
    </div>
    <div class="sc-card-meta">
      <strong>Grain:</strong> one row per session<br>
      <strong>Rows:</strong> 40 sessions, ~3,000 nested responses
    </div>
  </div>

  <div class="sc-card csv">
    <div class="sc-icon-box">
      <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"/><polyline points="14 2 14 8 20 8"/></svg>
      <div class="sc-csv-badge">CSV</div>
    </div>
    <div>
      <div class="sc-card-name">participant_lookup.csv</div>
      <div class="sc-card-source">CSV file from HR (you upload via Lakeflow Designer)</div>
      <p class="sc-card-cols"><strong>Columns:</strong> id, email, department, job_title, region</p>
    </div>
    <div class="sc-card-meta">
      <strong>Grain:</strong> one row per participant<br>
      <strong>Rows:</strong> 598
    </div>
  </div>

</div>

<div class="sc-job">
  <div class="sc-job-title">Your job:</div>
  <ul class="sc-job-list">
    <li>Today this data is <strong>bronze</strong>. Raw, mixed data types, nested arrays, no joins between tables.</li>
    <li>Build <strong>silver</strong> tables that are clean, typed, and joinable.</li>
    <li>Build <strong>gold</strong> KPIs that answer real business questions about attendance, ratings, and engagement.</li>
    <li>Do it all with <strong>no code</strong>, using Lakeflow Designer.</li>
  </ul>
</div>

</div>



## C. Preview the Tables Using Catalog Explorer

One of the great things about Databricks is that you can explore data **without writing any code**. We will do all of the table exploration in this section through the Catalog Explorer UI.

Let's get familiar with what's in each table before we build our visual data prep Lakeflow Designer.

### C1. Open Catalog Explorer in a new tab

1. In your Workspace sidebar, select the **Catalog** icon ![Catalog Icon](./Includes/images/common/catalog-icon.png).

2. Right-click on your **learn_no_code_etl_designer_USERNAME** catalog and select **Open in Catalog Explorer**.
   - Keeping this notebook and Catalog Explorer side-by-side makes the rest of the section easier to follow.

3. In Catalog Explorer, select the **designer_meeting** schema.

4. Confirm you see three tables:
   - **feedback_bronze**
   - **participants_bronze**
   - **sessions_bronze**

5. At the top of Catalog Explorer, select a SQL warehouse from the warehouse selector.

6. Leave **Catalog Explorer** open.

### C2. Explore sessions_bronze

This table is **one row per session**. It tells us what each session was, when it ran, who presented, and how long it was planned to run.

**In Catalog Explorer:**

1. Click **sessions_bronze**.

2. Click the **Sample Data** tab to preview the table.

3. While you look at the sample data, notice the following:

   | Column | What to notice |
   |---|---|
   | **meeting_uuid** | The primary key. Every other table joins back to this column. |
   | **how_name**, **presenter**, **topic_category**, **difficulty_level** | Session title, presenter, and grouping fields. |
   | **event_date** | Looks like a date, but it is stored as a **string**. We will fix that in silver. |
   | **_rescued_data** | Empty. This is the Auto Loader column that captures malformed rows. We will drop it in silver. |

4. Now ask the following question about the data to explore it.

   > How many sessions are there per topic_category?

5. Databricks generates the SQL and runs it for you. Read the result.

**NOTE:** Feel free to continue exploring the data if you want additional details.

### C3. Explore participants_bronze

This table is **one row per participant, per session**. It is the attendance log.

**In Catalog Explorer:**

1. Navigate to **participants_bronze** > **Sample Data**.

2. While you look at the sample data, notice the following:

   | Column | What to notice |
   |---|---|
   | **meeting_uuid** | Joins back to **sessions_bronze**. |
   | **id** | The participant ID. Joins to the participant lookup CSV you will upload later. |
   | **join_time**, **leave_time** | Look like timestamps, but they are stored as **strings**. We will cast them in silver. |
   | **_rescued_data** | Empty. This is the Auto Loader column that captures malformed rows. We will drop it in silver. |

3. Now ask the following questions about the data to explore it.


> Check for null IDs in the data and show all rows.

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Warning - Data Issues
  </strong>
  <div style="color:#333;">

#### Issue 1 - Null IDs
The source data has **20** rows with a `NULL` id (no participant identity). We will drop them in silver because they cannot be joined back to any attendee or feedback record.

  </div>
</div>


> Calculate the total duration in minutes each participant was in the session. The join_time and leave_time columns contain date and time values stored as text, for example 2025-11-03T12:04:27.000+00:00. Convert these text values to timestamps before calculating the duration. Include the original join_time and leave_time values and sort by duration ascending. Do not use a custom datetime pattern.

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Warning - Data Issues
  </strong>
  <div style="color:#333;">

#### Issue 2 - Invalid Join Time
Notice that the first 23 rows have a negative **duration_in_minutes** value.

There is an issue with our source data causing this (the **leave_time** is recorded earlier than the **join_time** for these attendees).

We will take this into account during our **bronze -> silver** data preparation. We will convert the **duration_in_minutes** to `NULL` for these rows so we can keep the participants in our scenario without skewing downstream averages.

##### Example Output (Can Vary with AI)

| participant_id | meeting_uuid | duration_in_minutes |
|---|---|---:|
| `+vnj6eUbRXil6g+A5UQ2Aw==` | `4iUQF0z3Rs+SG6H6Ap0cdQ==` | -38479020 |
| `O2f4GoghTqu89VTtEkJTKw==` | `MUVmi3YYTNyigYwwrmD4VQ==` | -38479020 |
| `iayLedmCQ2evbeN3EaBgvw==` | `6uqID4Q6RNOs9BF+ysV86Q==` | -38479020 |
| `...` | `...` | `...` |

  </div>
</div>


### C4. Explore feedback_bronze (A STRING column containing a JSON array of objects.)

This one is different. Each row is one session, and the **responses** column holds **every survey answer** for that session as nested data.

**In Catalog Explorer:**

1. Click **feedback_bronze**.

2. Click the **Sample Data** tab to preview the table. Then click the small expand arrow on a **responses** cell to peek inside one row.

3. While you look at the sample data, notice the following:

   | Column | What to notice |
   |---|---|
   | **meeting_uuid** | Identifies which session this feedback belongs to. Joins back to **sessions_bronze**. |
   | **responses** | A string column that holds a JSON array of objects. Each object in the array has 4 fields: **answer**, **question**, **question_id**, **respondent_id**. |
   | Row count | Only **40 rows**, one per session. But each row contains dozens of nested responses. |
   | What we will do in silver | Flatten the responses so every individual answer becomes its own row. |

   The **responses** column contains **5 distinct questions**:

   | Question | Answer type |
   |---|---|
   | Rate the session content. | Numeric (1-5) |
   | Rate the speaker. | Numeric (1-5) |
   | What did you like about the content in this session? | Free text |
   | What could have been improved? | Free text |
   | What additional topics would you like to get a deep dive on? | Free text |

### C5. Preview the Participant Lookup CSV


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Information
  </strong>
  <div style="color:#333;">
  
We are providing the CSV file with the course content, but for the workshop **pretend you have this file sitting on your laptop from another internal source**.

In the next notebook, you will upload it into Databricks through the Lakeflow Designer file upload UI, the same way you would upload it from your local machine.
  </div>
</div>


The final table is created from a **CSV file provided** by HR.
- This file is in the `participant_lookup` folder. We will simulate uploading it to Lakeflow Designer later.
<br></br>
1. In the main folder, navigate to the `participant_lookup` folder > `how_participant_lookup_week_1.csv`.

**Quick information about the participant lookup CSV file:**

| Property | Value |
|---|---|
| Source | CSV file in workspace (you will upload it via Lakeflow Designer next) |
| Grain | One row per participant |
| Join key | **id** -> **participants_bronze.id** |
| Columns | **id**, **email**, **department**, **job_title**, **region** |
| Delimiter | comma |
| Row count | 598 |

## D. Project Overview

Now that you know the data, here is the medallion workflow you will build in the next notebook using **Lakeflow Designer**.

<!--
  =============================================================================
  HOW TO GRAY OUT NODES
  =============================================================================
  Add class="dim" (alongside any existing classes) to any element you want to
  fade into the background.
  =============================================================================
-->
<div class="pipe-wrap">

<style>
.pipe-wrap {
    max-width: 920px;
    margin: 0 auto;
    font-family: sans-serif;
    color: #0b2026;
    padding: 8px;
    box-sizing: border-box;
}

/* Add class="dim" to any element to fade it into the background */
.dim {
    opacity: 0.30;
    filter: grayscale(100%);
}

.pipe-header {
    background: #1B5162;
    color: white;
    border-radius: 10px;
    padding: 22px 26px;
    text-align: center;
    margin-bottom: 22px;
    box-shadow: 0 2px 8px rgba(27,49,57,0.10);
}

.pipe-title {
    font-size: 20pt;
    font-weight: 700;
    color: white;
    margin: 0 0 10px 0;
}

.pipe-subtitle {
    font-size: 13pt;
    color: white;
    opacity: 0.94;
    line-height: 1.5;
    margin: 0;
    display: inline-block;
    text-align: left;
    padding-left: 24px;
}

.pipe-subtitle li { margin-bottom: 4px; }
.pipe-subtitle li:last-child { margin-bottom: 0; }
.pipe-subtitle strong { color: #FFAB00; font-weight: 700; }

.pipe-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    column-gap: 14px;
    row-gap: 8px;
    align-items: stretch;
    justify-items: stretch;
}

.pipe-arrow-cell {
    display: flex;
    justify-content: center;
    align-items: center;
}

.pipe-node {
    width: 100%;
    height: 100%;
    background: #F9F7F4;
    border-radius: 8px;
    box-shadow: 0 2px 8px rgba(27,49,57,0.06);
    padding: 14px 10px 12px 10px;
    text-align: center;
    position: relative;
    box-sizing: border-box;
    display: flex;
    flex-direction: column;
    justify-content: flex-start;
}

.pipe-node::before {
    content: "";
    position: absolute;
    top: 0;
    left: 0;
    width: 100%;
    height: 6px;
    border-top-left-radius: 8px;
    border-top-right-radius: 8px;
}

.pipe-node.source::before      { background: #00A972; }
.pipe-node.bronze::before      { background: #CD7F32; }
.pipe-node.silver::before      { background: #90A5B1; }
.pipe-node.gold::before        { background: #FFAB00; }

/* Bonus card: dashed border instead of solid shadow */
.pipe-node.bonus {
    border: 2px dashed #B57F00;
    background: rgba(255, 171, 0, 0.06);
    box-shadow: none;
}

.bonus-badge {
    position: absolute;
    top: -10px;
    right: 8px;
    background: #B57F00;
    color: white;
    font-size: 8pt;
    font-weight: 700;
    letter-spacing: 1px;
    padding: 3px 8px;
    border-radius: 4px;
    text-transform: uppercase;
    z-index: 2;
}

.pipe-icon {
    width: 30px;
    height: 30px;
    margin: 4px auto 6px auto;
    display: block;
}

.pipe-node.source .pipe-icon      { color: #00A972; }
.pipe-node.bronze .pipe-icon      { color: #CD7F32; }
.pipe-node.silver .pipe-icon      { color: #5A6F77; }
.pipe-node.gold .pipe-icon        { color: #B57F00; }

.pipe-node-label {
    font-size: 9pt;
    font-weight: 700;
    letter-spacing: 1.5px;
    text-transform: uppercase;
    margin: 0 0 4px 0;
}

.pipe-node.source .pipe-node-label      { color: #00A972; }
.pipe-node.bronze .pipe-node-label      { color: #CD7F32; }
.pipe-node.silver .pipe-node-label      { color: #5A6F77; }
.pipe-node.gold .pipe-node-label        { color: #B57F00; }

.pipe-node-name {
    font-size: 11pt;
    font-weight: 700;
    color: #0b2026;
    word-break: break-word;
    line-height: 1.3;
    margin: 0 0 6px 0;
}

.pipe-node-desc {
    font-size: 12pt;
    font-weight: 400;
    color: #5E7077;
    line-height: 1.4;
    margin: 0;
}

.pipe-arrow {
    width: 22px;
    height: 28px;
    color: #5E7077;
}

.volume-wrapper {
    border: 2px dashed #1B5162;
    border-radius: 10px;
    padding: 10px 8px 10px 8px;
    background: rgba(27,81,98,0.04);
    box-sizing: border-box;
    height: 100%;
    display: flex;
    flex-direction: column;
}

.volume-label {
    font-size: 8.5pt;
    font-weight: 700;
    letter-spacing: 1.5px;
    text-transform: uppercase;
    color: #1B5162;
    text-align: center;
    margin: 0 0 8px 0;
}

.upstream-wrapper {
    border: 2px dashed #5E7077;
    border-radius: 10px;
    padding: 10px 14px;
    background: rgba(94,112,119,0.06);
    box-sizing: border-box;
    height: 100%;
    display: flex;
    flex-direction: column;
}

.upstream-label {
    font-size: 10pt;
    font-weight: 700;
    letter-spacing: 1.5px;
    text-transform: uppercase;
    color: #5E7077;
    text-align: center;
    margin: 0 0 8px 0;
}

.upstream-body {
    display: flex;
    align-items: center;
    gap: 14px;
    flex: 1;
    padding: 4px 8px;
}

.upstream-icon {
    width: 36px;
    height: 36px;
    color: #5E7077;
    flex-shrink: 0;
}

.upstream-title {
    font-size: 14pt;
    font-weight: 700;
    color: #0b2026;
    margin-bottom: 4px;
    line-height: 1.3;
}

.upstream-desc {
    font-size: 12pt;
    color: #5E7077;
    line-height: 1.4;
}

.silver-to-gold {
    display: flex;
    flex-direction: column;
    align-items: center;
    margin: 14px 0 18px 0;
}

.silver-to-gold svg {
    width: 30px;
    height: 38px;
    color: #5E7077;
}

.silver-to-gold-caption {
    font-size: 13pt;
    font-weight: 700;
    color: #1B5162;
    margin-top: 4px;
}

.gold-grid {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    column-gap: 14px;
    margin-top: 0;
}
</style>

<div class="pipe-header">
  <h3 class="pipe-title">The Visual Workflow You Will Build</h3>
  <ul class="pipe-subtitle">
    <li>Import a local CSV file into a volume.</li>
    <li>Bronze gets cleaned into <strong>silver</strong>.</li>
    <li>Two gold tables are derived from silver. A bonus AI-powered table is offered as homework.</li>
  </ul>
</div>

<div class="pipe-grid">

  <!-- Row 1: Source (CSV) + Upstream ETL banner -->
  <div style="grid-column: 1; grid-row: 1;">
    <div class="volume-wrapper">
      <div class="volume-label">Manual File Upload to a Volume</div>
      <div class="pipe-node source">
        <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><path d="M5 2h11l4 4v15a1 1 0 0 1-1 1H5a1 1 0 0 1-1-1V3a1 1 0 0 1 1-1z"/><path d="M16 2v4h4"/><text x="12" y="19" text-anchor="middle" font-family="sans-serif" font-size="10" font-weight="700" stroke="none" fill="currentColor">CSV</text></svg>
        <div class="pipe-node-label">CSV File</div>
        <p class="pipe-node-name">how_participant_lookup_week_1.csv</p>
        <p class="pipe-node-desc">Uploaded participant directory</p>
      </div>
    </div>
  </div>

  <div style="grid-column: 2 / span 3; grid-row: 1;">
    <div class="upstream-wrapper">
      <div class="upstream-label">Upstream workflow &middot; Runs on a Schedule</div>
      <div class="upstream-body">
        <svg class="upstream-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.6" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true">
          <path d="M21 12a9 9 0 1 1-3.51-7.12"/>
          <polyline points="21 4 21 9 16 9"/>
        </svg>
        <div>
          <div class="upstream-title">Source workflow managed by another team</div>
          <div class="upstream-desc">Another team ingests the meeting source data as bronze tables on a schedule, refreshing the tables with new sessions, attendees, and feedback. You will keep silver and gold updated.</div>
        </div>
      </div>
    </div>
  </div>

  <!-- Row 2: arrows from CSV and Upstream ETL down to bronze -->
  <div class="pipe-arrow-cell" style="grid-column: 1; grid-row: 2;">
    <svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg>
  </div>
  <div class="pipe-arrow-cell" style="grid-column: 2; grid-row: 2;">
    <svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg>
  </div>
  <div class="pipe-arrow-cell" style="grid-column: 3; grid-row: 2;">
    <svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg>
  </div>
  <div class="pipe-arrow-cell" style="grid-column: 4; grid-row: 2;">
    <svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg>
  </div>

  <!-- Row 3: Bronze tables -->
  <div style="grid-column: 1; grid-row: 3;">
    <div class="pipe-node bronze">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Bronze</div>
      <p class="pipe-node-name">participant_lookup_bronze</p>
      <p class="pipe-node-desc">Lookup ingested from the CSV</p>
    </div>
  </div>
  <div style="grid-column: 2; grid-row: 3;">
    <div class="pipe-node bronze">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Bronze</div>
      <p class="pipe-node-name">sessions_bronze</p>
      <p class="pipe-node-desc">Raw session metadata, one row per session</p>
    </div>
  </div>
  <div style="grid-column: 3; grid-row: 3;">
    <div class="pipe-node bronze">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Bronze</div>
      <p class="pipe-node-name">participants_bronze</p>
      <p class="pipe-node-desc">Attendance log, one row per attendee per session</p>
    </div>
  </div>
  <div style="grid-column: 4; grid-row: 3;">
    <div class="pipe-node bronze">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Bronze</div>
      <p class="pipe-node-name">feedback_bronze</p>
      <p class="pipe-node-desc">Survey responses stored as a JSON array per session</p>
    </div>
  </div>

  <!-- Row 4: Bronze -> Silver arrows -->
  <div class="pipe-arrow-cell" style="grid-column: 1; grid-row: 4;"><svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg></div>
  <div class="pipe-arrow-cell" style="grid-column: 2; grid-row: 4;"><svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg></div>
  <div class="pipe-arrow-cell" style="grid-column: 3; grid-row: 4;"><svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg></div>
  <div class="pipe-arrow-cell" style="grid-column: 4; grid-row: 4;"><svg class="pipe-arrow" viewBox="0 0 20 30" fill="none" aria-hidden="true"><line x1="10" y1="0" x2="10" y2="22" stroke="currentColor" stroke-width="2.5" stroke-linecap="round"/><polygon points="3,21 17,21 10,30" fill="currentColor"/></svg></div>

  <!-- Row 5: Silver tables -->
  <div style="grid-column: 1; grid-row: 5;">
    <div class="pipe-node silver">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Silver</div>
      <p class="pipe-node-name">participant_lookup_silver</p>
      <p class="pipe-node-desc">Email and region split into parts</p>
    </div>
  </div>
  <div style="grid-column: 2; grid-row: 5;">
    <div class="pipe-node silver">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Silver</div>
      <p class="pipe-node-name">sessions_silver</p>
      <p class="pipe-node-desc">Date typed, how_name renamed to session_name</p>
    </div>
  </div>
  <div style="grid-column: 3; grid-row: 5;">
    <div class="pipe-node silver">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Silver</div>
      <p class="pipe-node-name">participants_silver</p>
      <p class="pipe-node-desc">Timestamps typed, duration_in_minutes added</p>
    </div>
  </div>
  <div style="grid-column: 4; grid-row: 5;">
    <div class="pipe-node silver">
      <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
      <div class="pipe-node-label">Silver</div>
      <p class="pipe-node-name">feedback_silver</p>
      <p class="pipe-node-desc">JSON exploded, one row per response</p>
    </div>
  </div>

</div>

<!-- Single centered arrow between silver and gold -->
<div class="silver-to-gold">
  <svg width="30" height="38" viewBox="0 0 30 38" fill="none" aria-hidden="true"><line x1="15" y1="2" x2="15" y2="26" stroke="currentColor" stroke-width="3" stroke-linecap="round"/><polygon points="4,26 26,26 15,38" fill="currentColor"/></svg>
  <div class="silver-to-gold-caption">Join and aggregate</div>
</div>

<!-- Gold row: 2 main gold tables + 1 bonus -->
<div class="gold-grid">
  <div class="pipe-node gold">
    <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
    <div class="pipe-node-label">Gold</div>
    <p class="pipe-node-name">gold_session_summary</p>
    <p class="pipe-node-desc">Session metadata with attendance metrics</p>
  </div>
  <div class="pipe-node gold">
    <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
    <div class="pipe-node-label">Gold</div>
    <p class="pipe-node-name">gold_feedback_summary</p>
    <p class="pipe-node-desc">Session-level average ratings</p>
  </div>
  <div class="pipe-node gold bonus">
    <div class="bonus-badge">Bonus / Homework</div>
    <svg class="pipe-icon" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round" stroke-linecap="round" aria-hidden="true"><rect x="3" y="4" width="18" height="16" rx="1"/><line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="14" x2="21" y2="14"/><line x1="9" y1="4" x2="9" y2="20"/><line x1="15" y1="4" x2="15" y2="20"/></svg>
    <div class="pipe-node-label">Gold</div>
    <p class="pipe-node-name">gold_feedback_sentiment</p>
    <p class="pipe-node-desc">Free-form responses tagged with ai_sentiment</p>
  </div>
</div>

</div>



&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>